# ML Testing & MLOps: Data Contracts and Statistical Performance Monitoring for a Credit Risk Pipeline

**Dataset:** `sklearn.datasets.fetch_openml(name="credit-g")` — the real-world **German
Credit Data** set from OpenML/UCI: 1,000 actual loan applications, 20 mixed
numeric/categorical features (duration, credit amount, age, job, housing, purpose, ...),
binary target `class` (`good` / `bad` credit risk).

**Problem statement.** A bank's credit-risk model service scores loan applications in
daily batches. Two things can quietly go wrong in production that a single "it works on
my test set" check will miss: (1) an incoming batch can violate the assumptions the model
was trained under — a column goes missing, a feature's values drift out of their historical
range, a categorical value nobody has ever seen appears — and (2) the model's real-world
performance can degrade in a way that isn't obviously explained by any single bad column.
Your job, coming from an SQA background, is to treat both as testable contracts rather than
things you'd only notice by eyeballing a dashboard.

**What you should produce:**

1. A **data contract** derived from the training data (expected dtypes, numeric ranges,
   categorical domains) and a `validate_batch()` function that checks a new batch against it
   and returns a list of concrete violations (not just True/False).
2. A baseline model with a **bootstrap confidence interval** on its holdout ROC-AUC — a
   single point estimate is not enough to decide "did performance actually drop, or is this
   just batch-to-batch noise?"
3. A simulated **stream of incoming batches**, where a few specific batches have a
   *controlled, known* problem injected (missing-data spike, feature drift, a batch where the
   model genuinely gets worse without any schema problem at all).
4. A `monitor_batch()` function that runs the contract check and a bootstrap-AUC check on
   each batch and flags a **statistically significant** performance drop (CI-overlap test,
   not "the number went down").
5. Two **CI-style test functions** (`assert`-based, printing a pass/fail line per batch,
   like a test report) that a real pipeline could run before letting a batch reach
   production scoring — including one that specifically catches the dangerous case: a
   performance drop with *no* corresponding data-quality explanation.

---

## Starter cells

Fill in every `# TODO`. Don't look at the solution until you've tried each step yourself.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

RNG = np.random.default_rng(42)

### 1. Load the data

`credit-g` ships as a DataFrame with the target column named `class` (values `"good"` /
`"bad"`). Recode it to an int target where `1 = bad credit risk` (the minority, higher-stakes
class).

In [ ]:
credit = fetch_openml(name="credit-g", version=1, as_frame=True)
df = credit.frame.copy()
df["class"] = (df["class"] == "bad").astype(int)  # 1 = bad credit risk

print(df.shape)
df.head()

### 2. Build a data contract from the training data

Split off a holdout set first — the contract must be derived only from training data, the
same way you'd only know your production assumptions from what you trained on.

In [ ]:
TRAIN_DF, HOLDOUT_DF = train_test_split(
    df, test_size=0.3, random_state=42, stratify=df["class"]
)


def build_contract(train_df: pd.DataFrame) -> dict:
    """TODO: derive a validation contract from train_df.

    Return a dict keyed by column name (excluding "class"), where each value describes:
      - for numeric columns: {"dtype": "numeric", "min": ..., "max": ...}
        (consider padding the observed min/max slightly so you don't flag every
        legitimately-slightly-larger-than-anything-seen-before value)
      - for categorical columns: {"dtype": "categorical", "categories": set_of_allowed_values}
    """
    # TODO
    raise NotImplementedError


CONTRACT = build_contract(TRAIN_DF)

### 3. Validate a batch against the contract

In [ ]:
def validate_batch(batch_df: pd.DataFrame, contract: dict,
                    missing_thresh: float = 0.05, dup_thresh: float = 0.02) -> list[dict]:
    """TODO: check batch_df against `contract`. Return a list of violation dicts, each like
        {"column": "duration", "rule": "range", "detail": "..."}

    Rules to check, at minimum:
      - missing or unexpected columns vs. the contract
      - per-column missing-value rate above `missing_thresh`
      - numeric values outside the contract's [min, max]
      - categorical values outside the contract's allowed categories
      - duplicate-row rate above `dup_thresh`
    """
    # TODO
    raise NotImplementedError

### 4. Train a baseline model and get a bootstrap CI for its AUC

A single holdout AUC number can't tell you whether a later batch's AUC is a *real* drop or
just noise. Bootstrap resampling gives you a confidence interval to compare against.

In [ ]:
def train_baseline(train_df: pd.DataFrame):
    """TODO: build and fit a sklearn Pipeline (ColumnTransformer + LogisticRegression) on
    train_df's features vs. "class". Scale numeric columns, one-hot encode categoricals.
    Return the fitted pipeline."""
    # TODO
    raise NotImplementedError


def bootstrap_auc_ci(model, X: pd.DataFrame, y: pd.Series,
                      n_boot: int = 1000, ci: float = 0.95, rng=RNG) -> tuple[float, float, float]:
    """TODO: compute the point-estimate ROC-AUC of `model` on (X, y), then bootstrap-resample
    (X, y) with replacement n_boot times, score each resample, and return
    (point_estimate, lower_bound, upper_bound) for the given confidence level."""
    # TODO
    raise NotImplementedError


model = train_baseline(TRAIN_DF)
baseline_point, baseline_lo, baseline_hi = bootstrap_auc_ci(
    model, HOLDOUT_DF.drop(columns="class"), HOLDOUT_DF["class"]
)
print(f"baseline AUC: {baseline_point:.3f}  95% CI [{baseline_lo:.3f}, {baseline_hi:.3f}]")

### 5. Simulate an incoming batch stream — with controlled, injected failure modes

> We split the real holdout set into sequential batches, then deliberately corrupt a couple
> of them. This is still real data underneath; only specific cells/columns are perturbed.
> The injection is what gives you *ground truth* — you need to know which batches SHOULD be
> flagged in order to tell whether your monitoring code actually works.

In [ ]:
def make_batch_stream(holdout_df: pd.DataFrame, n_batches: int = 6, rng=RNG) -> list[pd.DataFrame]:
    """TODO: shuffle and split holdout_df into n_batches roughly-equal DataFrames, then:
      - in batch index 2: null out ~20% of the values in the "duration" column
        (missing-data spike)
      - in batch index 4: shift the "age" column by several standard deviations for every
        row in that batch (feature drift)
      - in batch index 5: randomly permute the "credit_amount" column's values across rows
        (destroys a real predictive signal without changing dtypes, ranges, or categories —
        the schema still looks perfectly fine)
    Return the list of 6 batch DataFrames.
    """
    # TODO
    raise NotImplementedError


BATCHES = make_batch_stream(HOLDOUT_DF)

### 6. Monitor each batch: contract check + statistical performance check

In [ ]:
def monitor_batch(batch_df: pd.DataFrame, contract: dict, model, baseline_ci: tuple) -> dict:
    """TODO: for one batch, return a report dict with:
      - "contract_violations": output of validate_batch(batch_df, contract)
      - "auc_point", "auc_lo", "auc_hi": bootstrap_auc_ci(...) on this batch
      - "performance_alert": True iff this batch's CI does NOT overlap baseline_ci
        (a statistically significant drop — not just "the point estimate is lower")
    """
    # TODO
    raise NotImplementedError


reports = [monitor_batch(b, CONTRACT, model, (baseline_lo, baseline_hi)) for b in BATCHES]

### 7. Turn it into a CI-style test suite

These should read like tests you'd actually wire into a pipeline: assert something, print a
pass/fail line per batch so a human (or CI log) can see exactly what happened.

In [ ]:
def test_no_critical_contract_violations(reports: list[dict], max_allowed: int = 0):
    """TODO: for each batch's report, print "[PASS]"/"[FAIL]" plus the violation count.
    Then assert that no batch has an unreasonable number of violations (pick and justify a
    threshold)."""
    # TODO
    raise NotImplementedError


def test_no_significant_unexplained_performance_drop(reports: list[dict]):
    """TODO: the dangerous case isn't "performance dropped AND the contract caught something"
    (that's at least explained) — it's "performance dropped significantly with a perfectly
    clean contract". For each batch, print "[PASS]"/"[FAIL]" and assert that no batch is in
    that dangerous state."""
    # TODO
    raise NotImplementedError


test_no_critical_contract_violations(reports)
test_no_significant_unexplained_performance_drop(reports)

---

## Solution

*(scroll down when you're ready — try it yourself first)*

<details>
<summary>Click to reveal solution</summary>

```python
def build_contract(train_df: pd.DataFrame) -> dict:
    contract = {}
    for col in train_df.columns:
        if col == "class":
            continue
        s = train_df[col]
        if pd.api.types.is_numeric_dtype(s):
            lo, hi = s.min(), s.max()
            pad = 0.05 * (hi - lo) if hi > lo else 1.0
            contract[col] = {"dtype": "numeric", "min": lo - pad, "max": hi + pad}
        else:
            contract[col] = {"dtype": "categorical", "categories": set(s.dropna().unique())}
    return contract


def validate_batch(batch_df, contract, missing_thresh=0.05, dup_thresh=0.02):
    violations = []
    expected_cols = set(contract.keys())
    actual_cols = set(batch_df.columns) - {"class"}

    for missing_col in expected_cols - actual_cols:
        violations.append({"column": missing_col, "rule": "missing_column",
                            "detail": "column absent from batch"})
    for extra_col in actual_cols - expected_cols:
        violations.append({"column": extra_col, "rule": "unexpected_column",
                            "detail": "column not in contract"})

    for col, rule in contract.items():
        if col not in batch_df.columns:
            continue
        s = batch_df[col]

        miss_rate = s.isna().mean()
        if miss_rate > missing_thresh:
            violations.append({"column": col, "rule": "missing_rate",
                                "detail": f"{miss_rate:.1%} missing (> {missing_thresh:.0%})"})

        non_null = s.dropna()
        if rule["dtype"] == "numeric":
            bad = non_null[(non_null < rule["min"]) | (non_null > rule["max"])]
            if len(bad) > 0:
                violations.append({"column": col, "rule": "range",
                                    "detail": f"{len(bad)} values outside "
                                              f"[{rule['min']:.1f}, {rule['max']:.1f}]"})
        else:
            unexpected = set(non_null.unique()) - rule["categories"]
            if unexpected:
                violations.append({"column": col, "rule": "domain",
                                    "detail": f"unexpected categories: {unexpected}"})

    dup_rate = batch_df.duplicated().mean()
    if dup_rate > dup_thresh:
        violations.append({"column": "__row__", "rule": "duplicates",
                            "detail": f"{dup_rate:.1%} duplicate rows"})

    return violations


def train_baseline(train_df):
    X, y = train_df.drop(columns="class"), train_df["class"]
    num_cols = X.select_dtypes(include="number").columns.tolist()
    cat_cols = X.select_dtypes(exclude="number").columns.tolist()

    pre = ColumnTransformer([
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ])
    pipe = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=1000))])
    pipe.fit(X, y)
    return pipe


def bootstrap_auc_ci(model, X, y, n_boot=1000, ci=0.95, rng=RNG):
    X = X.reset_index(drop=True)
    y = y.reset_index(drop=True)
    n = len(X)
    point = roc_auc_score(y, model.predict_proba(X)[:, 1])

    scores = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        yb = y.iloc[idx]
        if yb.nunique() < 2:
            continue
        proba = model.predict_proba(X.iloc[idx])[:, 1]
        scores.append(roc_auc_score(yb, proba))

    alpha = (1 - ci) / 2
    lo, hi = np.quantile(scores, [alpha, 1 - alpha])
    return point, lo, hi


def make_batch_stream(holdout_df, n_batches=6, rng=RNG):
    shuffled = holdout_df.sample(frac=1, random_state=42)
    chunks = [c.copy() for c in np.array_split(shuffled, n_batches)]

    idx = chunks[2].sample(frac=0.2, random_state=1).index
    chunks[2].loc[idx, "duration"] = np.nan

    chunks[4]["age"] = chunks[4]["age"] + 4 * holdout_df["age"].std()

    chunks[5]["credit_amount"] = rng.permutation(chunks[5]["credit_amount"].values)

    return chunks


def monitor_batch(batch_df, contract, model, baseline_ci):
    violations = validate_batch(batch_df, contract)
    point, lo, hi = bootstrap_auc_ci(
        model, batch_df.drop(columns="class"), batch_df["class"], n_boot=500
    )
    base_lo, base_hi = baseline_ci
    overlaps = not (hi < base_lo or lo > base_hi)
    return {
        "contract_violations": violations,
        "auc_point": point, "auc_lo": lo, "auc_hi": hi,
        "performance_alert": not overlaps,
    }


def test_no_critical_contract_violations(reports, max_allowed=0):
    ok = True
    for i, r in enumerate(reports):
        n = len(r["contract_violations"])
        status = "PASS" if n <= max_allowed else "FAIL"
        ok &= (n <= max_allowed + 2)
        print(f"[{status}] batch {i}: {n} contract violation(s)")
    assert ok, "a batch had an unreasonable number of contract violations"


def test_no_significant_unexplained_performance_drop(reports):
    ok = True
    for i, r in enumerate(reports):
        unexplained = r["performance_alert"] and not r["contract_violations"]
        status = "FAIL" if unexplained else "PASS"
        ok &= not unexplained
        print(f"[{status}] batch {i}: alert={r['performance_alert']}, "
              f"violations={len(r['contract_violations'])}")
    assert ok, "found a performance drop with no corresponding data-quality explanation"
```

**Why this works:**

- The contract catches batch 2 (missing-data spike) and batch 4 (feature drift) directly —
  these are the "obvious" failures an SQA-style schema/range check is built for.
- Batch 5 is the interesting one: `credit_amount` is permuted, so its dtype, range, and
  categorical domain all still look completely normal — the contract passes. But the
  model's real AUC on that batch genuinely drops, because the feature-target relationship is
  broken. The bootstrap-CI overlap check is what catches *that*, instead of chasing noise in
  a single point estimate.
- `test_no_significant_unexplained_performance_drop` encodes the actual SQA judgment call:
  a flagged batch with a data-quality explanation is "known and handled"; a flagged batch
  with a clean contract is the silent failure mode worth paging someone for.

</details>